# Design Matrix

This script loads data from the different participants and merges them to one big design matrix.

- flags outliers based on placement accuracy and adds them to the bad_epochs column
- removes unused columns
- add here any manual data cleaning for specific participants if necessary

In [27]:
import os
import pandas as pd

# path = 'P:\\Lukas_Gehrke\\NAH\\behavior\\5_single-subject-EEG-analysis'
path = '/Users/lukasgehrke/data/NAH/data/5_single-subject-EEG-analysis/'

In [28]:
# Initialize an empty behaviorframe to store the aggregated behavior
data = pd.DataFrame()

# Loop over the ids and append behavior to the general behaviorframe
for id in range(7, 15):

    if id == 13 or id == 8:
        continue
    
    pID = 'sub-' + "%01d" % (id)
    behavior = pd.read_csv(os.path.join(path, pID + '/behavior_s' + str(id) + '.csv'), delimiter=';')
    behavior = behavior.select_dtypes(exclude=['object'])

    # remove outlier trials based on super bad placement
    # Calculate Q1 (25th percentile) and Q3 (75th percentile) for AccuracyCm
    Q1_AccuracyCm = behavior['AccuracyCm'].quantile(0.25)
    Q3_AccuracyCm = behavior['AccuracyCm'].quantile(0.75)

    # Calculate IQR for AccuracyCm
    IQR_AccuracyCm = Q3_AccuracyCm - Q1_AccuracyCm

    # Define the lower and upper bounds for outliers in AccuracyCm
    lower_bound_AccuracyCm = Q1_AccuracyCm - 1.5 * IQR_AccuracyCm
    upper_bound_AccuracyCm = Q3_AccuracyCm + 1.5 * IQR_AccuracyCm

    # Calculate Q1 (25th percentile) and Q3 (75th percentile) for fix_delay
    Q1_fix_delay = behavior['fix_delay'].quantile(0.25)
    Q3_fix_delay = behavior['fix_delay'].quantile(0.75)

    # Calculate IQR for fix_delay
    IQR_fix_delay = Q3_fix_delay - Q1_fix_delay

    # Define the lower and upper bounds for outliers in fix_delay
    lower_bound_fix_delay = Q1_fix_delay - 1.5 * IQR_fix_delay
    upper_bound_fix_delay = Q3_fix_delay + 1.5 * IQR_fix_delay

    # Mark all rows where the condition is met in the column bad_epochs as 1
    behavior.loc[(behavior['AccuracyCm'] < lower_bound_AccuracyCm) | (behavior['AccuracyCm'] > upper_bound_AccuracyCm), 'bad_epoch'] = 1

    # print both behavior and behavior_filt shape with some text
    print('Behavior shape:', behavior.shape)

    # behavior = behavior.groupby(['hapticProfile']).mean()
    behavior['id'] = id
    data = pd.concat([data, behavior])

data.reset_index(inplace=True)
data = data.replace({'hapticProfile': {0: 'baseline', 1: 'sound', 2: 'vibration', 3: 'vibration+sound'}})
# drop columns ISI, latency, urevent, duration
data = data.drop(columns=['index', 'ISI', 'latency', 'urevent', 'duration'])
# save the data
data.to_csv(os.path.join(path, 'behavior_all.csv'), sep=';', index=False)
print(data.shape)

Behavior shape: (152, 10)
Behavior shape: (140, 10)
Behavior shape: (140, 10)
Behavior shape: (140, 10)
Behavior shape: (140, 10)
Behavior shape: (140, 10)
(852, 7)
